In [ ]:
!pip install keras-tuner

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

In [3]:
print(tf.__version__)

2.6.0


In [4]:
fashion_mnist=keras.datasets.fashion_mnist

In [ ]:
(train_images,train_labels),(test_images,test_labels)=fashion_mnist.load_data()

In [ ]:
train_images=train_images/255.0
test_images=test_images/255.0

In [10]:
train_images[0].shape

(28, 28, 1)

In [9]:
train_images=train_images.reshape(len(train_images),28,28,1)
test_images=test_images.reshape(len(test_images),28,28,1)

In [11]:
def build_model(hp):  
  model = keras.Sequential([
    keras.layers.Conv2D(
        filters=hp.Int('conv_1_filter', min_value=32, max_value=128, step=16),
        kernel_size=hp.Choice('conv_1_kernel', values = [3,5]),
        activation='relu',
        input_shape=(28,28,1)
    ),
    keras.layers.Conv2D(
        filters=hp.Int('conv_2_filter', min_value=32, max_value=64, step=16),
        kernel_size=hp.Choice('conv_2_kernel', values = [3,5]),
        activation='relu'
    ),
    keras.layers.Flatten(),
    keras.layers.Dense(
        units=hp.Int('dense_1_units', min_value=32, max_value=128, step=16),
        activation='relu'
    ),
    keras.layers.Dense(10, activation='softmax')
  ])
  
  model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[1e-2, 1e-3])),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
  
  return model

In [12]:
from kerastuner import RandomSearch
from kerastuner.engine.hyperparameters import HyperParameters

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  """Entry point for launching an IPython kernel.


In [13]:
tuner_search=RandomSearch(build_model,
                          objective='val_accuracy',
                          max_trials=5,directory='/content',project_name="Mnist Fashion")

In [14]:
tuner_search.search(train_images,train_labels,epochs=3,validation_split=0.1)

Trial 5 Complete [00h 00m 47s]
val_accuracy: 0.8669999837875366

Best val_accuracy So Far: 0.9139999747276306
Total elapsed time: 00h 06m 36s
INFO:tensorflow:Oracle triggered exit


In [18]:
model=tuner_search.get_best_models(num_models=1)[0]

In [19]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 26, 26, 32)        320       
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 22, 22, 48)        38448     
_________________________________________________________________
flatten (Flatten)            (None, 23232)             0         
_________________________________________________________________
dense (Dense)                (None, 96)                2230368   
_________________________________________________________________
dense_1 (Dense)              (None, 10)                970       
Total params: 2,270,106
Trainable params: 2,270,106
Non-trainable params: 0
_________________________________________________________________


In [20]:
model.fit(train_images, train_labels, epochs=10, validation_split=0.1, initial_epoch=3)

Epoch 4/10
1688/1688 [==============================] - 15s 9ms/step - loss: 0.1287 - accuracy: 0.9503 - val_loss: 0.2649 - val_accuracy: 0.9140
Epoch 5/10
1688/1688 [==============================] - 14s 8ms/step - loss: 0.0897 - accuracy: 0.9676 - val_loss: 0.2936 - val_accuracy: 0.9132
Epoch 6/10
1688/1688 [==============================] - 14s 8ms/step - loss: 0.0614 - accuracy: 0.9772 - val_loss: 0.3483 - val_accuracy: 0.9055
Epoch 7/10
1688/1688 [==============================] - 13s 8ms/step - loss: 0.0449 - accuracy: 0.9839 - val_loss: 0.4009 - val_accuracy: 0.9130
Epoch 8/10
1688/1688 [==============================] - 13s 8ms/step - loss: 0.0367 - accuracy: 0.9867 - val_loss: 0.4466 - val_accuracy: 0.9088
Epoch 9/10
1688/1688 [==============================] - 13s 8ms/step - loss: 0.0275 - accuracy: 0.9900 - val_loss: 0.5182 - val_accuracy: 0.9093
Epoch 10/10
1688/1688 [==============================] - 13s 8ms/step - loss: 0.0232 - accuracy: 0.9915 - val_loss: 0.4908 - val_a